# Spark execution
Today we will dive into the execution internals of spark. We will be reading some (fake) transactions and perform some transformations on them. While doing these transformations, we aim to get more insights into how spark works. We will check wide and narrow transformations, shuffles and spark plans to understand what is happening behind the code.

Spark needs to run with java 17. Let's firs downgrade by running the following code in the terminal below `sdk install java 17.0.18-ms -y` and type `y` to confirm version 17 to be set as default. Confirm by running `java --version`

## First we will import the pyspark library and create a spark session

In [2]:
from pyspark.sql import SparkSession, functions as sf

In [3]:
spark = SparkSession.builder.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/20 14:25:09 WARN Utils: Your hostname, codespaces-b2bb9b, resolves to a loopback address: 127.0.0.1; using 10.0.0.249 instead (on interface eth0)
26/05/20 14:25:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/20 14:25:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Now we will read the data and we will show the first records

In [4]:
df = spark.read.csv("file:///workspaces/spark-workshop/transactions.csv", header=True)

Run the cell below, what do you see?

In [5]:
df

DataFrame[amount: string, currency: string, payer_account_number: string, beneficiary_account_number: string, payer_country_code: string, beneficiary_country_code: string, transaction_timestamp: string]

Now run the show command.

In [6]:
df.show()

+-------+--------+--------------------+--------------------------+------------------+------------------------+---------------------+
| amount|currency|payer_account_number|beneficiary_account_number|payer_country_code|beneficiary_country_code|transaction_timestamp|
+-------+--------+--------------------+--------------------------+------------------+------------------------+---------------------+
|3383.64|     EUR|  HU28EXDV9600133890|      FR71GWUW379402654...|                HU|                      FR| 2025-07-10T05:06:54Z|
| 769.08|     DKK|  GB93ORDM8495931034|      GB10HDMI525534192...|                GB|                      GB| 2025-02-03T17:58:29Z|
|4627.42|     CAD|IE29BZKM139537672...|      RO33XSNS532871012...|                IE|                      RO| 2025-02-13T10:45:32Z|
|8858.91|     JPY|AU87XDVR514627048...|      SE77GELY880957015430|                AU|                      SE| 2025-01-09T22:01:18Z|
| 224.86|     USD|IE08YRYE782489638346|      PL85ULOQ133150983...|   

Do you recognize what just happened? Remember lazy evaluation and actions?

We called show, an action, and spark started to execute.

Open the ports section below and right click on the forwarded address from port 4040. Click on open in browser. The spark UI will open in a new browser tab. Inspect this UI to see what spark has done.

## Make a column where we want to check if this is a domestic transaction

In [7]:
df = df.withColumn(
    "is_domestic",
    sf.col("payer_country_code") == sf.col("beneficiary_country_code")
)

## We want to only keep eur and usd transactions, create a filter for that

In [8]:
df_eur_usd = df.where(
    sf.col("currency").isin(["USD", "EUR"])
)

## Let's check how spark executes this

In [9]:
df_eur_usd.explain()

== Physical Plan ==
*(1) Project [amount#17, currency#18, payer_account_number#19, beneficiary_account_number#20, payer_country_code#21, beneficiary_country_code#22, transaction_timestamp#23, (payer_country_code#21 = beneficiary_country_code#22) AS is_domestic#54]
+- *(1) Filter currency#18 IN (USD,EUR)
   +- FileScan csv [amount#17,currency#18,payer_account_number#19,beneficiary_account_number#20,payer_country_code#21,beneficiary_country_code#22,transaction_timestamp#23] Batched: false, DataFilters: [currency#18 IN (USD,EUR)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/workspaces/spark-workshop/transactions.csv], PartitionFilters: [], PushedFilters: [In(currency, [EUR,USD])], ReadSchema: struct<amount:string,currency:string,payer_account_number:string,beneficiary_account_number:strin...




Let's do a no-op (fake) write to call an action, such that we can also visualise the actual execution plan in the spark ui

In [10]:
df_eur_usd.write.format("noop").mode("overwrite").save() 

What do you notice? Now check the spark UI, the SQL/Dataframe tab in particular. What do you see here? How does it relate to the explain?

## What happens when we filter on only domestic transactions?

In [11]:
df_domestic = df.where(
    sf.col("is_domestic")
)

In [12]:
df_domestic.explain()

== Physical Plan ==
*(1) Project [amount#17, currency#18, payer_account_number#19, beneficiary_account_number#20, payer_country_code#21, beneficiary_country_code#22, transaction_timestamp#23, (payer_country_code#21 = beneficiary_country_code#22) AS is_domestic#54]
+- *(1) Filter ((isnotnull(payer_country_code#21) AND isnotnull(beneficiary_country_code#22)) AND (payer_country_code#21 = beneficiary_country_code#22))
   +- FileScan csv [amount#17,currency#18,payer_account_number#19,beneficiary_account_number#20,payer_country_code#21,beneficiary_country_code#22,transaction_timestamp#23] Batched: false, DataFilters: [isnotnull(payer_country_code#21), isnotnull(beneficiary_country_code#22), (payer_country_code#21..., Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/workspaces/spark-workshop/transactions.csv], PartitionFilters: [], PushedFilters: [IsNotNull(payer_country_code), IsNotNull(beneficiary_country_code)], ReadSchema: struct<amount:string,currency:string,payer_account_number:s

Let's do another noop write

In [13]:
df_domestic.write.format("noop").mode("overwrite").save() 

and check the spark UI again, what's happening?

## We want to convert the payer country codes to full country names

Run the code below to prevent spark from converting your join to broadcast hash joins

In [28]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.shuffle.partitions", 10)

First, read in the conversion table. Then perform a left join.

In [29]:
country_df = spark.read.csv("file:///workspaces/spark-workshop/iso_country_codes.csv", header=True)
df_smj = df.join(
    country_df,
    how="left",
    on=df["payer_country_code"] == country_df["country_code"]
).drop("country_code").withColumnRenamed("country_name", "payer_country_name")

In [30]:
df_smj.explain()

== Physical Plan ==
*(5) Project [amount#17, currency#18, payer_account_number#19, beneficiary_account_number#20, payer_country_code#21, beneficiary_country_code#22, transaction_timestamp#23, is_domestic#54, country_name#184 AS payer_country_name#186]
+- *(5) SortMergeJoin [payer_country_code#21], [country_code#183], LeftOuter
   :- *(2) Sort [payer_country_code#21 ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(payer_country_code#21, 10), ENSURE_REQUIREMENTS, [plan_id=645]
   :     +- *(1) Project [amount#17, currency#18, payer_account_number#19, beneficiary_account_number#20, payer_country_code#21, beneficiary_country_code#22, transaction_timestamp#23, (payer_country_code#21 = beneficiary_country_code#22) AS is_domestic#54]
   :        +- FileScan csv [amount#17,currency#18,payer_account_number#19,beneficiary_account_number#20,payer_country_code#21,beneficiary_country_code#22,transaction_timestamp#23] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileI

What do you notice? You can also check the spark ui.

In [31]:
df_bhj = df.join(
    sf.broadcast(country_df),
    how="left",
    on=df["payer_country_code"] == country_df["country_code"]
).drop("country_code").withColumnRenamed("country_name", "payer_country_name")


In [32]:
df_bhj.explain()

== Physical Plan ==
*(2) Project [amount#17, currency#18, payer_account_number#19, beneficiary_account_number#20, payer_country_code#21, beneficiary_country_code#22, transaction_timestamp#23, is_domestic#54, country_name#184 AS payer_country_name#187]
+- *(2) BroadcastHashJoin [payer_country_code#21], [country_code#183], LeftOuter, BuildRight, false
   :- *(2) Project [amount#17, currency#18, payer_account_number#19, beneficiary_account_number#20, payer_country_code#21, beneficiary_country_code#22, transaction_timestamp#23, (payer_country_code#21 = beneficiary_country_code#22) AS is_domestic#54]
   :  +- FileScan csv [amount#17,currency#18,payer_account_number#19,beneficiary_account_number#20,payer_country_code#21,beneficiary_country_code#22,transaction_timestamp#23] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/workspaces/spark-workshop/transactions.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<amount:string,currency:strin

Does this plan look different than before? Which one do you think will be faster?

Let's do a noop write

In [33]:
df_smj.write.format("noop").mode("overwrite").save()
df_bhj.write.format("noop").mode("overwrite").save() 

And check the spark ui

In [34]:
df_smj.groupBy("payer_country_code").count().show(100, False)

+------------------+-----+
|payer_country_code|count|
+------------------+-----+
|AT                |38   |
|CZ                |38   |
|US                |48   |
|IE                |45   |
|NO                |33   |
|AU                |48   |
|FR                |53   |
|PL                |51   |
|CA                |45   |
|DE                |34   |
|HU                |57   |
|IT                |40   |
|PT                |50   |
|BE                |32   |
|CH                |40   |
|DK                |47   |
|GB                |38   |
|LU                |40   |
|RO                |37   |
|SE                |40   |
|ES                |47   |
|BG                |45   |
|NL                |54   |
+------------------+-----+



## Now we would like to know the total amounts transferred for each currency

In [40]:
total_currency = df.groupBy(
    "currency"
).agg(sf.sum("amount"))

In [42]:
total_currency.show(100, False)

+--------+------------------+
|currency|sum(amount)       |
+--------+------------------+
|JPY     |180501.85         |
|USD     |150220.13999999998|
|CZK     |107805.70000000003|
|NOK     |184630.59000000003|
|GBP     |154512.24000000002|
|RON     |164541.28999999995|
|EUR     |133610.13         |
|PLN     |146438.16999999998|
|AUD     |168126.17000000004|
|SEK     |170259.54000000004|
|DKK     |166036.32999999993|
|BGN     |164519.40000000002|
|CHF     |245965.6          |
|CAD     |213561.25000000003|
|HUF     |157473.73         |
+--------+------------------+



In [37]:
total_currency.explain()

== Physical Plan ==
*(2) HashAggregate(keys=[currency#18], functions=[sum(cast(amount#17 as double))])
+- Exchange hashpartitioning(currency#18, 10), ENSURE_REQUIREMENTS, [plan_id=954]
   +- *(1) HashAggregate(keys=[currency#18], functions=[partial_sum(cast(amount#17 as double))])
      +- FileScan csv [amount#17,currency#18] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/workspaces/spark-workshop/transactions.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<amount:string,currency:string>




What do you notice?